# 04 -- Phase 3: Photometry & Zero Point

Detect sources, measure them with the ERR plane, derive a filter-wise zero
point, flux-calibrate to Janskys, and write an error-carrying catalog.

The zero point is calibrated by cross-matching against an online reference
catalog in a priority cascade: **APASS -> Pan-STARRS -> SDSS** (first hit
wins). This step needs outbound internet on the node.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG
# ============================================================
# One shared module rather than this cell copied into six notebooks, so a
# path is changed once and the notebooks cannot drift apart.
# Override any path with an environment variable; see workshop_config.py.
import importlib, os, sys

_here = os.path.dirname(os.path.abspath('workshop_config.py'))
if _here not in sys.path:
    sys.path.insert(0, _here)

# Reloaded, not merely imported. A kernel that imported workshop_config before
# the file was edited keeps serving the cached module, and the first name added
# since then fails much further down as a bare NameError -- which is exactly how
# `raw_frames()` broke for anyone whose kernel predated it.
import workshop_config
importlib.reload(workshop_config)
from workshop_config import *   # noqa: F403  (RAW_DIR, WORK_DIR, PHASE*_DIR, ...)

require_dataset()   # fails now, with the command that fixes it, not later
os.makedirs(WORK_DIR, exist_ok=True)
show_config()

In [ ]:
import glob, os
assert os.path.isdir(PHASE2_DIR), "No phase2 directory -- run notebook 03 (Phase 2) first."
masters = sorted(glob.glob(os.path.join(PHASE2_DIR, "Master_*.fits")))
assert masters, "No Master_*.fits in PHASE2_DIR -- run notebook 03 (Phase 2) first."
print(f"{len(masters)} master stack(s) in {PHASE2_DIR}")

## Run Phase 3

In [ ]:
from cassa_photometry.config import load_config
from cassa_photometry.phase3_photometry import run as run_p3
cfg = load_config()
run_p3(PHASE2_DIR, default_band='R', outdir=PHASE3_DIR, config=cfg)

## The zero point lives in the flux-calibrated header

In [ ]:
import glob, os
from astropy.io import fits

# Phase 3 writes `_fluxcal.fits` ONLY when it measured a zero point. Without one
# it still writes the catalog -- with instrumental magnitudes and MAG_ISO = NaN.
# That is a real outcome to read, not a crash, so handle both.
fluxcals = sorted(glob.glob(os.path.join(PHASE3_DIR, '*_fluxcal.fits')))
catalogs = sorted(glob.glob(os.path.join(PHASE3_DIR, '*_catalog.csv')))

if fluxcals:
    print(os.path.basename(fluxcals[0]))
    h = fits.getheader(fluxcals[0])
    for k in ['MAGZERO', 'MAGZERR', 'NZPSTARS', 'FLUXCAL', 'BUNIT']:
        print(f'{k:>9} = {h.get(k)}')
else:
    print("No *_fluxcal.fits in PHASE3_DIR: Phase 3 could not measure a zero point.\n")
    print("The zero point comes from cross-matching detected stars against a reference")
    print("catalog, so it fails when the frame yields no usable stars -- which is a")
    print("statement about the image, not about this step. The Phase 3 log above gives")
    print("the reason; `No stars detected` means the star finder came back empty on a")
    print("master that a person would say has stars in it. Work backwards: look at the")
    print("Phase 2 master, then at the Phase 1 frames that went into it.\n")
    print(f"{len(catalogs)} catalog(s) were still written, with instrumental magnitudes:")
    for c in catalogs:
        print('   ', os.path.basename(c))
    print(f"\nLog: {os.path.join(PHASE3_DIR, 'cassa_photometry.log')}")

## Explore the source catalog

In [ ]:
import glob, os
import numpy as np
import pandas as pd

cat = pd.read_csv(sorted(glob.glob(os.path.join(PHASE3_DIR, '*_catalog.csv')))[0])
print(len(cat), 'sources')

# Everything below plots a magnitude. Without a zero point MAG_ISO is NaN
# throughout, but MAG_INST -- the same measurement without the additive
# constant -- is not. Choose the scale here so one failed zero point does not
# empty every plot in the notebook.
CALIBRATED = bool(np.isfinite(cat['MAG_ISO']).any())
MAG_COL, MAGERR_COL = ('MAG_ISO', 'MAGERR_ISO') if CALIBRATED else ('MAG_INST', 'MAGERR_INST')
MAG_LABEL = 'magnitude' if CALIBRATED else 'instrumental magnitude (no ZP)'

if not CALIBRATED:
    print("\nNo zero point: MAG_ISO is NaN for every source, so the cells below use")
    print("MAG_INST instead. The two differ by one additive constant, so every shape")
    print("here -- error against brightness, the number counts, the SNR curve -- is")
    print("unchanged. Only the zero of the scale is arbitrary, so no number from this")
    print("catalog can be compared with another image's.")
cat.head()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(cat[MAG_COL], cat[MAGERR_COL], s=6, alpha=0.4)
ax[0].set_xlabel(MAG_LABEL); ax[0].set_ylabel(MAGERR_COL); ax[0].set_title('error vs mag')
ax[1].scatter(cat[MAG_COL], cat['SNR'], s=6, alpha=0.4)
ax[1].set_yscale('log'); ax[1].set_xlabel(MAG_LABEL); ax[1].set_ylabel('SNR')
ax[1].set_title('SNR vs mag')
plt.tight_layout(); plt.show()

## (Optional) Independent verification against reference catalogs

In [ ]:
from cassa_photometry.phase3_photometry.verify import run as verify_run
verify_run(PHASE3_DIR)

### Exercise 1 -- check the zero-point arithmetic yourself

Recompute `MAG_ISO` and `MAGERR_ISO` from `FLUX_ISO`, `FLUXERR_ISO` and the
header, using $m = -2.5\log_{10}(F) + \mathrm{ZP}$ and
$\delta m = \sqrt{(1.0857\,\delta F/F)^2 + \mathrm{MAGZERR}^2}$.

Pair the catalog with *its own* `_fluxcal.fits` by base name -- a zero point
belongs to one image, and the two `glob` calls above can land on different
filters.

| Find | Expected |
|---|---|
| largest residual against the catalog | `TBD` mag |
| which term dominates for the brightest star | `TBD` |

In [ ]:
import glob, os
import numpy as np
import pandas as pd
from astropy.io import fits

# Fill in the blanks marked TODO. Everything else is scaffolding.
# A zero point belongs to ONE image, so pair the catalog with its own fluxcal by
# base name -- the globs earlier in this notebook can land on different filters.
cat_path = sorted(glob.glob(os.path.join(PHASE3_DIR, '*_catalog.csv')))[0]
base = os.path.basename(cat_path).replace('_catalog.csv', '')
fc_path = os.path.join(PHASE3_DIR, base + '_fluxcal.fits')

if not os.path.exists(fc_path):
    print(f"Skipped: no {os.path.basename(fc_path)}, so this image has no zero point")
    print("to verify. The Phase 3 log says why the cross-match failed.")
else:
    paired = pd.read_csv(cat_path)
    ph = fits.getheader(fc_path)
    MAGZERO, MAGZERR = ph['MAGZERO'], ph['MAGZERR']
    flux, flux_err = paired['FLUX_ISO'], paired['FLUXERR_ISO']

    mag_predicted = FILL_IN    # TODO 1: -2.5 log10(F) + ZP
    err_predicted = FILL_IN    # TODO 2: sqrt((1.0857 dF/F)^2 + MAGZERR^2)

    shot = 1.0857 * flux_err / flux
    brightest = int(np.argmin(paired['MAG_ISO'].values))

    print(f"{base}\n  MAGZERO = {MAGZERO:.4f} +/- {MAGZERR:.4f} "
          f"(from {ph.get('NZPSTARS')} stars)\n")
    print(f"largest residual, MAG_ISO    : "
          f"{np.abs(paired['MAG_ISO'] - mag_predicted).max():.2e} mag")
    print(f"largest residual, MAGERR_ISO : "
          f"{np.abs(paired['MAGERR_ISO'] - err_predicted).max():.2e} mag\n")
    print(f"brightest star: measurement {shot.values[brightest]:.4f} mag, "
          f"zero point {MAGZERR:.4f} mag")
    print("  -> " + ("the zero point dominates; more exposure time will not help it"
                     if MAGZERR > shot.values[brightest] else
                     "the measurement dominates; more exposure time still helps"))

### Exercise 2 -- the magnitude of your assigned star

Your instructor gives you one star, either by its catalog `NUMBER` or by its
RA/Dec. Report its brightness with an honest uncertainty, and say which half of
the error budget dominates: the measurement, or the zero point.

Fill in `ASSIGNED_NUMBER` **or** `ASSIGNED_RADEC` in the cell below.

| Find | Expected |
|---|---|
| magnitude of your star | `TBD` mag |
| its 1-sigma uncertainty | `TBD` mag |
| the dominant error term | `TBD` |

Report `MAG_APER`, not `MAG_ISO`. `MAG_ISO` is isophotal -- it captures a
brightness-dependent fraction of a star -- so an aperture-derived zero point
applied to it tilts the scale rather than shifting it. `MAG_APER` is measured in
the very aperture the zero point was, which is why the pipeline's Phase 3 log
tells you to use it.

A magnitude quoted without an uncertainty is not a measurement, and one quoted
to more digits than its uncertainty supports is a claim you cannot defend --
round the value to the precision the error allows.

In [ ]:
# ============================================================
#  YOUR ASSIGNED STAR -- fill in ONE of these, as your instructor assigned it.
# ============================================================
ASSIGNED_NUMBER = None        # e.g. 7                       (catalog NUMBER)
ASSIGNED_RADEC = None         # e.g. (278.97970, -23.81821)  (degrees)
MATCH_ARCSEC = 3.0            # how close a source must be to count as yours
#
# Then fill in the blanks marked TODO further down. The rest is scaffolding.

import glob
import os
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits

star = None
if ASSIGNED_NUMBER is not None:
    hits = cat[cat['NUMBER'] == ASSIGNED_NUMBER]
    if len(hits) == 0:
        raise ValueError(f"No source NUMBER={ASSIGNED_NUMBER}; the catalog holds "
                         f"{int(cat['NUMBER'].min())}-{int(cat['NUMBER'].max())}.")
    star = hits.iloc[0]
elif ASSIGNED_RADEC is not None:
    # Match by position, and refuse a match that is too far away: silently
    # reporting the nearest source is how you measure the wrong star.
    target = SkyCoord(*ASSIGNED_RADEC, unit='deg')
    sources = SkyCoord(cat['ALPHA_J2000'].to_numpy(),
                       cat['DELTA_J2000'].to_numpy(), unit='deg')
    separations = target.separation(sources).arcsec
    nearest = int(np.argmin(separations))
    if separations[nearest] > MATCH_ARCSEC:
        raise ValueError(f"Nearest source is {separations[nearest]:.1f} arcsec away, "
                         f"beyond the {MATCH_ARCSEC} arcsec tolerance. Wrong field?")
    star = cat.iloc[nearest]
    print(f"Matched {separations[nearest]:.2f} arcsec from the assigned position.\n")
else:
    print("Set ASSIGNED_NUMBER or ASSIGNED_RADEC above. The catalog holds:\n")
    print(cat[['NUMBER', 'ALPHA_J2000', 'DELTA_J2000', MAG_COL, MAGERR_COL]]
          .head(10).to_string(index=False))

# The zero point was measured in the aperture, so MAG_APER is the column that
# is exactly consistent with it. Without a zero point it is NaN like every other
# calibrated column, and only the instrumental magnitude survives.
STAR_MAG, STAR_ERR = ('MAG_APER', 'MAGERR_APER') if CALIBRATED else ('MAG_INST', 'MAGERR_INST')
FLUX, FLUXERR = ('FLUX_APER', 'FLUXERR_APER') if CALIBRATED else ('FLUX_ISO', 'FLUXERR_ISO')

if star is not None:
    mag, err = float(star[STAR_MAG]), float(star[STAR_ERR])

    # Split the budget. The measurement term shrinks with exposure time; the
    # zero-point term is common to every star in the image and does not.
    measurement = FILL_IN    # TODO 1: this star's 1.0857 dF/F in magnitudes, from FLUX/FLUXERR
    zp_term = np.nan
    if CALIBRATED:
        base = os.path.basename(sorted(glob.glob(
            os.path.join(PHASE3_DIR, '*_catalog.csv')))[0]).replace('_catalog.csv', '')
        fc = os.path.join(PHASE3_DIR, base + '_fluxcal.fits')
        if os.path.exists(fc):
            zp_term = float(fits.getheader(fc).get('MAGZERR', np.nan))

    # Quote no more precision than the uncertainty supports.
    digits = FILL_IN         # TODO 2: decimals to print, one past the error's leading digit

    print(f"Star NUMBER {int(star['NUMBER'])} at "
          f"RA {star['ALPHA_J2000']:.5f}, Dec {star['DELTA_J2000']:+.5f}\n")
    print(f"  {STAR_MAG:<12} = {mag:.{digits}f} +/- {err:.{digits}f}"
          f"{'' if CALIBRATED else '   (instrumental -- no zero point in this run)'}")
    print(f"  SNR          = {float(star['SNR']):.1f}")
    print(f"  {FLUX:<12} = {float(star[FLUX]):.4g} +/- "
          f"{float(star[FLUXERR]):.3g}\n")

    print(f"  error budget: measurement 1.0857 dF/F = {measurement:.4f} mag")
    if np.isfinite(zp_term):
        print(f"                zero point   MAGZERR  = {zp_term:.4f} mag")
        dominant = 'the zero point' if zp_term > measurement else 'the measurement'
        print(f"\n  {dominant} dominates. "
              + ("Longer exposures will not help this star until the zero point "
                 "itself improves." if zp_term > measurement else
                 "More exposure time on this star still buys precision."))
    else:
        print("                zero point   MAGZERR  = not available (no flux calibration)")
        print("\n  Without a zero point only the measurement term exists, so this")
        print("  uncertainty is incomplete: it says how well the flux was measured,")
        print("  not how well the brightness is known.")